In [ ]:
import os, zipfile, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers, models
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.utils import image_dataset_from_directory
from keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("TF Version   :", tf.__version__)
print("Keras Version:", keras.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("All imports successful!")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
import shutil, os, random

def find_dataset_path(base='/kaggle/input'):
    for root, dirs, files in os.walk(base):
        if any(d in dirs for d in ['train', 'val', 'test']):
            return root
    return None

SRC  = find_dataset_path()

if SRC is None:
    print(" Could not auto-detect dataset. Available paths:")
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level <= 4:
            print('  ' * level + root)
    raise FileNotFoundError("Please set SRC manually from the paths above.")

print(f" Dataset found at: {SRC}")

SEED = 42
DEST = '/kaggle/working/dataset'

if not os.path.exists(DEST):
    print("Copying dataset to working directory...")
    shutil.copytree(SRC, DEST)
    print("Dataset copied!")
else:
    print("Already copied!")

TRAIN_DIR = os.path.join(DEST, 'train')
VAL_DIR   = os.path.join(DEST, 'val')

train_classes = set(sorted(os.listdir(TRAIN_DIR)))
val_classes   = set(sorted(os.listdir(VAL_DIR)))
missing_in_val = train_classes - val_classes

print(f"\nMissing classes in val: {len(missing_in_val)}")

random.seed(SEED)
for cls in missing_in_val:
    src = os.path.join(TRAIN_DIR, cls)
    dst = os.path.join(VAL_DIR,   cls)
    os.makedirs(dst, exist_ok=True)
    imgs  = sorted(os.listdir(src))
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs) * 0.15))
    for img in imgs[:n_val]:
        shutil.copy2(
            os.path.join(src, img),
            os.path.join(dst, img)
        )
    print(f"  Fixed: {cls}")

print(f"\nTrain classes : {len(os.listdir(TRAIN_DIR))}")
print(f"Val   classes : {len(os.listdir(VAL_DIR))}")

In [ ]:
IMG_SIZE        = (224, 224)
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 10
EPOCHS_FINETUNE = 20

DATASET_PATH = '/kaggle/working/dataset'
OUTPUT_PATH  = '/kaggle/working'

TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR   = os.path.join(DATASET_PATH, 'val')
TEST_DIR  = os.path.join(DATASET_PATH, 'test')

if not os.path.exists(TEST_DIR):
    TEST_DIR = VAL_DIR
    print("No test folder found — using val for test evaluation.")

classes     = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)
print(f"\nDetected {NUM_CLASSES} classes:")
for c in classes[:5]:
    print(f"  • {c}")
print("  ...")
print(f"\nDataset loaded from: {DATASET_PATH}")

In [ ]:
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle    = True,
    seed       = SEED,
    label_mode = 'categorical'
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle    = False,
    label_mode = 'categorical'
)

test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle    = False,
    label_mode = 'categorical'
)

CLASS_NAMES = train_ds.class_names
print(f"Training samples  : {train_ds.cardinality().numpy() * BATCH_SIZE}")
print(f"Validation samples: {val_ds.cardinality().numpy()   * BATCH_SIZE}")
print(f"Classes           : {len(CLASS_NAMES)}")

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

def prepare_train(images, labels):
    images = augmentation(images, training=True)
    images = preprocess_input(images)
    return images, labels

def prepare_eval(images, labels):
    images = preprocess_input(images)
    return images, labels

train_ds = (train_ds
            .map(prepare_train, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
val_ds   = (val_ds
            .map(prepare_eval, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
test_ds  = (test_ds
            .map(prepare_eval, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

print("Data pipeline ready!")

In [ ]:
for images, labels in train_ds.take(1):
    sample_images = images.numpy()
    sample_labels = labels.numpy()

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

for i in range(15):
    img = (sample_images[i] + 1.0) / 2.0
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(CLASS_NAMES[np.argmax(sample_labels[i])], fontsize=7)
    axes[i].axis('off')

plt.suptitle("Sample Training Images (after augmentation)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'sample_images.png'), dpi=150)
plt.show()


In [ ]:
def build_model(num_classes: int, trainable_backbone: bool = False):
    backbone = MobileNetV2(
        input_shape = (*IMG_SIZE, 3),
        include_top = False,
        weights     = 'imagenet'
    )
    backbone.trainable = trainable_backbone

    if trainable_backbone:
        for layer in backbone.layers[:-30]:
            layer.trainable = False
        for layer in backbone.layers[-30:]:
            layer.trainable = True

    inputs  = keras.Input(shape=(*IMG_SIZE, 3))
    x       = backbone(inputs, training=trainable_backbone)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='PlantMedic_MobileNetV2')
    return model, backbone

model, backbone = build_model(NUM_CLASSES, trainable_backbone=False)

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

model.summary()

trainable_count     = sum(tf.size(w).numpy() for w in model.trainable_weights)
non_trainable_count = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
print(f"\nTrainable params    : {trainable_count:,}")
print(f"Non-trainable params: {non_trainable_count:,}")


In [ ]:

class MetricsCallback(keras.callbacks.Callback):
    def __init__(self, val_dataset):
        super().__init__()
        self.val_dataset = val_dataset
        self.history_pr  = {'precision': [], 'recall': [], 'f1': []}

    def on_epoch_end(self, epoch, logs=None):
        y_true_list, y_pred_list = [], []
        for images, labels in self.val_dataset:
            preds = self.model.predict(images, verbose=0)
            y_pred_list.extend(np.argmax(preds,          axis=1))
            y_true_list.extend(np.argmax(labels.numpy(), axis=1))

        y_true = np.array(y_true_list)
        y_pred = np.array(y_pred_list)

        p  = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        r  = recall_score   (y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score       (y_true, y_pred, average='weighted', zero_division=0)

        self.history_pr['precision'].append(p)
        self.history_pr['recall'].append(r)
        self.history_pr['f1'].append(f1)

        if logs is not None:
            logs['val_precision'] = p
            logs['val_recall']    = r
            logs['val_f1']        = f1

        print(f"  → val_precision={p:.4f}  val_recall={r:.4f}  val_f1={f1:.4f}")

metrics_cb = MetricsCallback(val_ds)
print("MetricsCallback ready!")

In [ ]:
from PIL import Image
import os, struct

print("Deep scanning all images...")
corrupted = 0

for root, dirs, files in os.walk('/kaggle/working/dataset'):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            filepath = os.path.join(root, file)
            try:
                if os.path.getsize(filepath) == 0:
                    os.remove(filepath)
                    corrupted += 1
                    continue

                img = Image.open(filepath)
                img.load()
                img.close()

            except Exception as e:
                try:
                    os.remove(filepath)
                    corrupted += 1
                except:
                    pass

print(f"Deleted {corrupted} corrupted files!")
total = sum(len(f) for _, _, f in os.walk('/kaggle/working/dataset'))
print(f"Total clean files: {total}")

In [ ]:
train_ds_raw = image_dataset_from_directory(
    TRAIN_DIR,
    image_size = (224, 224),
    batch_size = 32,
    shuffle    = True,
    seed       = 42,
    label_mode = 'categorical'
)
val_ds_raw = image_dataset_from_directory(
    VAL_DIR,
    image_size = (224, 224),
    batch_size = 32,
    shuffle    = False,
    label_mode = 'categorical'
)

CLASS_NAMES = train_ds_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)

train_ds = (train_ds_raw.map(prepare_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))
val_ds   = (val_ds_raw.map(prepare_eval,   num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))
test_ds  = val_ds
metrics_cb = MetricsCallback(val_ds)

print(f"Classes : {NUM_CLASSES}")
print("Ready for Phase 1")

In [ ]:
CHECKPOINT_DIR = os.path.join(OUTPUT_PATH, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        os.path.join(CHECKPOINT_DIR, 'best_phase1.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    ),
    CSVLogger(os.path.join(CHECKPOINT_DIR, 'phase1_log.csv')),
    metrics_cb
]

print("=" * 60)
print("PHASE 1: Training head only  (backbone FROZEN)")
print("=" * 60)

history1 = model.fit(
    train_ds,
    epochs          = EPOCHS_FROZEN,
    validation_data = val_ds,
    callbacks       = callbacks_phase1
)

In [ ]:
import shutil, os

if os.path.exists('/kaggle/working/checkpoints/best_phase1.keras'):
    shutil.copy(
        '/kaggle/working/checkpoints/best_phase1.keras',
        '/kaggle/working/best_phase1_PERMANENT.keras'
    )
    print("Best Phase 1 checkpoint saved!")
else:
    print("No checkpoint found!")

In [ ]:
model_ft, _ = build_model(NUM_CLASSES, trainable_backbone=True)
phase1_model = keras.models.load_model('/kaggle/working/best_phase1_PERMANENT.keras')
model_ft.set_weights(phase1_model.get_weights())

model_ft.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
print("Phase 1 loaded — Starting Phase 2 directly!")

In [ ]:

print("=" * 60)
print("PHASE 2: Fine-tuning top 30 backbone layers + head")
print("=" * 60)

model_ft, _ = build_model(NUM_CLASSES, trainable_backbone=True)
model_ft.set_weights(model.get_weights())

model_ft.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

metrics_cb_ft = MetricsCallback(val_ds)

callbacks_phase2 = [
    ModelCheckpoint(
        os.path.join(CHECKPOINT_DIR, 'best_plantmedic.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=7,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=3, min_lr=1e-8, verbose=1
    ),
    CSVLogger(os.path.join(CHECKPOINT_DIR, 'phase2_log.csv')),
    metrics_cb_ft
]

history2 = model_ft.fit(
    train_ds,
    epochs          = EPOCHS_FINETUNE,
    validation_data = val_ds,
    callbacks       = callbacks_phase2
)

In [ ]:
def combine(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
split = len(history1.history['accuracy'])

axes[0,0].plot(combine(history1, history2, 'accuracy'),     label='Train')
axes[0,0].plot(combine(history1, history2, 'val_accuracy'), label='Val')
axes[0,0].axvline(x=split-1, color='gray', linestyle='--', label='Fine-tune start')
axes[0,0].set_title('Accuracy'); axes[0,0].legend(); axes[0,0].set_xlabel('Epoch')

axes[0,1].plot(combine(history1, history2, 'loss'),     label='Train')
axes[0,1].plot(combine(history1, history2, 'val_loss'), label='Val')
axes[0,1].axvline(x=split-1, color='gray', linestyle='--', label='Fine-tune start')
axes[0,1].set_title('Loss'); axes[0,1].legend(); axes[0,1].set_xlabel('Epoch')

all_f1 = metrics_cb.history_pr['f1'] + metrics_cb_ft.history_pr['f1']
axes[1,0].plot(all_f1, color='green', label='Val F1')
axes[1,0].axvline(x=split-1, color='gray', linestyle='--', label='Fine-tune start')
axes[1,0].set_title('F1 Score'); axes[1,0].legend(); axes[1,0].set_xlabel('Epoch')

all_p = metrics_cb.history_pr['precision'] + metrics_cb_ft.history_pr['precision']
all_r = metrics_cb.history_pr['recall']    + metrics_cb_ft.history_pr['recall']
axes[1,1].plot(all_p, label='Precision', color='orange')
axes[1,1].plot(all_r, label='Recall',    color='purple')
axes[1,1].set_title('Precision & Recall'); axes[1,1].legend()
axes[1,1].set_xlabel('Epoch')

plt.suptitle('PlantMedic — Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'training_curves.png'), dpi=150)
plt.show()
print("Training curves saved!")



In [ ]:
best_model = keras.models.load_model(
    os.path.join(CHECKPOINT_DIR, 'best_plantmedic.keras')
)

print("Evaluating on test set...")
y_true_list, y_pred_list = [], []

for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_pred_list.extend(np.argmax(preds,          axis=1))
    y_true_list.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score   (y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score       (y_true, y_pred, average='weighted', zero_division=0)

print("\n" + "=" * 50)
print("       FINAL TEST RESULTS — PlantMedic")
print("=" * 50)
print(f"  Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Precision     : {precision:.4f}")
print(f"  Recall        : {recall:.4f}")
print(f"  F1 Score      : {f1:.4f}")
print("=" * 50)

print("\nPer-Class Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm,
    annot       = True,
    fmt         = 'd',
    cmap        = 'YlOrRd',
    xticklabels = CLASS_NAMES,
    yticklabels = CLASS_NAMES,
    linewidths  = 0.3
)
plt.title('Confusion Matrix — PlantMedic (Test Set)', fontsize=14, pad=15)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label',      fontsize=11)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0,  fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'confusion_matrix.png'), dpi=150)
plt.show()
print("Confusion Matrix")



In [ ]:
import json

best_model.save(os.path.join(OUTPUT_PATH, 'plantmedic_final.keras'))
print(" Saved: plantmedic_final.keras")

# Save in H5 format
best_model.save(os.path.join(OUTPUT_PATH, 'plantmedic_final.h5'))
print(" Saved: plantmedic_final.h5")

# Save TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(os.path.join(OUTPUT_PATH, 'plantmedic.tflite'), 'wb') as f:
    f.write(tflite_model)
print(" Saved: plantmedic.tflite")

# Save class names
with open(os.path.join(OUTPUT_PATH, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print(" Saved: class_names.json")

print("\n" + "=" * 50)
print("ALL FILES SAVED TO /kaggle/working/")
print("=" * 50)
for fname in os.listdir(OUTPUT_PATH):
    fpath = os.path.join(OUTPUT_PATH, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f"   {fname}  —  {size_mb:.1f} MB")
print("\n All done! Download from Output panel on right side!")